<a href="https://colab.research.google.com/github/Ayushxsingh100/Amazon-ML-Challenge-2026/blob/main/notebooks/01_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1: Data Audit

The objective of this notebook is only to understand the data. It will produce your first five outputs:
1. Dataset row counts
2. Column information
3. Missing-value summary
4. Basic data-quality information
5. Duplicate statistics

Do not normalize or modify the data yet.

### Cell 1 — Install/import

In [6]:
!git clone https://github.com/Ayushxsingh100/Amazon-ML-Challenge-2026.git

Cloning into 'Amazon-ML-Challenge-2026'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 76 (delta 0), reused 0 (delta 0), pack-reused 72 (from 1)
Receiving objects: 100% (76/76), 1.01 GiB | 22.95 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Updating files: 100% (44/44), done.


In [7]:
from pathlib import Path

PROJECT_DIR = Path("/content/Amazon-ML-Challenge-2026")

TRAIN_DIR = PROJECT_DIR / "data" / "train"
TEST_DIR = PROJECT_DIR / "data" / "test"

print("Project:", PROJECT_DIR)
print("Train:", TRAIN_DIR)
print("Test:", TEST_DIR)

Project: /content/Amazon-ML-Challenge-2026
Train: /content/Amazon-ML-Challenge-2026/data/train
Test: /content/Amazon-ML-Challenge-2026/data/test


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Cell 2 — Connect Google Drive

In [8]:
print("Project exists:", PROJECT_DIR.exists())
print("Train exists:", TRAIN_DIR.exists())
print("Test exists:", TEST_DIR.exists())

Project exists: True
Train exists: True
Test exists: True


### Cell 3 — Set your data directory

In [9]:
print("\n========== TRAIN FILES ==========")

for f in sorted(TRAIN_DIR.iterdir()):
    print(f.name)

print("\n========== TEST FILES ==========")

for f in sorted(TEST_DIR.iterdir()):
    print(f.name)


========== TRAIN FILES ==========
train_ground_truth.tsv.part_aa
train_ground_truth.tsv.part_ab
train_source1.tsv.part_aa
train_source1.tsv.part_ab
train_source1.tsv.part_ac
train_source2.tsv.part_aa
train_source2.tsv.part_ab
train_source2.tsv.part_ac
train_source2.tsv.part_ad
train_source2.tsv.part_ae
train_source2.tsv.part_af
train_source3.tsv.part_aa
train_source3.tsv.part_ab
train_source3.tsv.part_ac
train_source3.tsv.part_ad
train_source3.tsv.part_ae
train_source3.tsv.part_af

========== TEST FILES ==========
test_source1.tsv.part_aa
test_source1.tsv.part_ab
test_source2.tsv.part_aa
test_source2.tsv.part_ab
test_source2.tsv.part_ac
test_source2.tsv.part_ad
test_source2.tsv.part_ae
test_source2.tsv.part_af
test_source3.tsv.part_aa
test_source3.tsv.part_ab
test_source3.tsv.part_ac
test_source3.tsv.part_ad
test_source3.tsv.part_ae
test_source3.tsv.part_af


### Cell 4 — Automatically discover the chunks

In [10]:
from pathlib import Path

print("========== PROJECT STRUCTURE ==========")

for path in sorted(PROJECT_DIR.iterdir()):
    print(path.name)

print("\n========== TRAIN FILES ==========")

for path in sorted(TRAIN_DIR.iterdir()):
    print(path.name)

print("\n========== TEST FILES ==========")

for path in sorted(TEST_DIR.iterdir()):
    print(path.name)

========== PROJECT STRUCTURE ==========
.git
.gitignore
README.md
data
notebooks
reconstruct_data.sh
src

========== TRAIN FILES ==========
train_ground_truth.tsv.part_aa
train_ground_truth.tsv.part_ab
train_source1.tsv.part_aa
train_source1.tsv.part_ab
train_source1.tsv.part_ac
train_source2.tsv.part_aa
train_source2.tsv.part_ab
train_source2.tsv.part_ac
train_source2.tsv.part_ad
train_source2.tsv.part_ae
train_source2.tsv.part_af
train_source3.tsv.part_aa
train_source3.tsv.part_ab
train_source3.tsv.part_ac
train_source3.tsv.part_ad
train_source3.tsv.part_ae
train_source3.tsv.part_af

========== TEST FILES ==========
test_source1.tsv.part_aa
test_source1.tsv.part_ab
test_source2.tsv.part_aa
test_source2.tsv.part_ab
test_source2.tsv.part_ac
test_source2.tsv.part_ad
test_source2.tsv.part_ae
test_source2.tsv.part_af
test_source3.tsv.part_aa
test_source3.tsv.part_ab
test_source3.tsv.part_ac
test_source3.tsv.part_ad
test_source3.tsv.part_ae
test_source3.tsv.part_af


### Cell 5 — Display discovered parts

In [11]:
SCRIPT = PROJECT_DIR / "reconstruct_data.sh"

print(SCRIPT.read_text())

#!/usr/bin/env bash
# Amazon ML Challenge 2026 - Reconstruct Data TSV files from git split parts
set -e

echo "=== Reconstructing Amazon ML Challenge 2026 Data Files ==="

reconstruct_file() {
    local target="$1"
    local dir=$(dirname "$target")
    local base=$(basename "$target")
    
    if [ -f "$target" ]; then
        echo "[EXISTS] $target is already present."
    else
        echo "[MERGING] Combining parts for $target ..."
        cat "${target}.part_"* > "$target"
        echo "[DONE] Successfully created $target ($(du -h "$target" | cut -f1))"
    fi
}

# Train sources
reconstruct_file "data/train/train_source1.tsv"
reconstruct_file "data/train/train_source2.tsv"
reconstruct_file "data/train/train_source3.tsv"
reconstruct_file "data/train/train_ground_truth.tsv"

# Test sources
reconstruct_file "data/test/test_source1.tsv"
reconstruct_file "data/test/test_source2.tsv"
reconstruct_file "data/test/test_source3.tsv"

echo "=== All data files ready! ==="



### Cell 6 — Check chunk sizes

In [13]:
def inspect_chunk_headers(directory, prefix):
    parts = sorted(directory.glob(prefix + ".tsv.part_*"))

    print(f"\n{'='*80}")
    print(prefix)
    print(f"{'='*80}")

    for part in parts:
        with open(part, "r", encoding="utf-8", errors="replace") as f:
            first_line = f.readline().strip()

        print(f"\n{part.name}")
        print(first_line[:300])


for prefix in [
    "train_source1",
    "train_source2",
    "train_source3",
    "train_ground_truth"
]:
    inspect_chunk_headers(TRAIN_DIR, prefix)


train_source1

train_source1.tsv.part_aa
entity_id	business_name	business_address	country

train_source1.tsv.part_ab
gar, Vijayawada, Krishna, Andhra Pradesh	India

train_source1.tsv.part_ac
nkcity Impex Private Limited	Flat No.11 Dwarka Apts 467/C Shivaji Nagar, Pune, Maharashtra	India

train_source2

train_source2.tsv.part_aa
entity_id	business_name	business_address	country

train_source2.tsv.part_ab
170 HIGLHAND AVENUE, HAMBURG, NY	US

train_source2.tsv.part_ac
9984094	Durham Womens Health Integrated Physicians Corp	3960 RIVER STONE ROAD, DURHAM, NC	US

train_source2.tsv.part_ad
-446836062	New Constructions  Hotel Private Limited	KHASRA NO 270, VILL, RASULPUR, MOJA BHARAMPUR, ETMADPUR, Uttar Pradesh	India

train_source2.tsv.part_ae
LE, KY	US

train_source2.tsv.part_af
मॉडर्न डेवलपर्स	Maharashtra, PUNE, KUMAR KSHITIJ FLAT B A-304SAHAKAR NAGAR -2	India

train_source3

train_source3.tsv.part_aa
entity_id	business_name	business_address	country

train_source3.tsv.part_ab
Fund Holdings	6